# 01 - EDA on UNSW-NB15

## 1. Loading the data

In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import pandas as pd

from netad.data import (
    DATA_DIR, FEATURES, CATEGORICAL, NUMERIC,
    load_unsw_frame, load_unsw, deduplicate, resolve_conflicts, overlap_with_train,
)

In [24]:
DATA_DIR, DATA_DIR.is_dir()

(WindowsPath('C:/Users/Gabi/Documents/Anomaly-detection-in-Network-traffic-using-ML/data/raw'),
 True)

In [25]:
len(FEATURES), len(CATEGORICAL), len(NUMERIC)

(42, 3, 39)

In [26]:
train_df = load_unsw_frame("train")
test_df = load_unsw_frame("test")
train_df.shape, test_df.shape

((175341, 45), (82332, 45))

First column should be `id` (no BOM).

In [27]:
train_df.columns[0]

'id'

In [28]:
X_train, y_train, y_train_multi = load_unsw("train")
X_test, y_test, y_test_multi = load_unsw("test")
X_train.shape, X_test.shape

((175341, 42), (82332, 42))

In [29]:
[c for c in ["id", "label", "attack_cat"] if c in X_train.columns]

[]

In [30]:
sorted(y_train.unique()), sorted(y_test.unique())

([np.int64(0), np.int64(1)], [np.int64(0), np.int64(1)])

`label` and `attack_cat` should agree: label 0 <=> Normal.

In [31]:
((y_train_multi == "Normal") == (y_train == 0)).all(), ((y_test_multi == "Normal") == (y_test == 0)).all()

(np.True_, np.True_)

In [32]:
round((y_train == 0).mean(), 3), round((y_test == 0).mean(), 3)

(np.float64(0.319), np.float64(0.449))

In [33]:
X_train[CATEGORICAL].dtypes

proto      str
service    str
state      str
dtype: object

### Duplicates and train/test overlap

In [34]:
train_dedup = deduplicate(train_df)
len(train_df), len(train_dedup), len(train_df) - len(train_dedup)

(175341, 107740, 67601)

In [35]:
in_train = overlap_with_train(train_df, test_df)
in_train.sum(), (~in_train).sum()

(np.int64(8541), np.int64(73791))

### Conflicting labels
Same feature vector, different label. Counted on raw train rows.

In [36]:
for target in ["label", "attack_cat"]:
    n_values = train_df.groupby(FEATURES)[target].transform("nunique")
    rows = train_df[n_values > 1]
    print(f"{target}: {len(rows)} rows ({len(rows) / len(train_df):.1%}), "
          f"{len(rows[FEATURES].drop_duplicates())} vectors")

label: 940 rows (0.5%), 229 vectors
attack_cat: 30528 rows (17.4%), 1772 vectors


In [37]:
conflict = train_df.groupby(FEATURES)["attack_cat"].transform("nunique") > 1
pd.DataFrame({
    "rows": train_df["attack_cat"].value_counts(),
    "in_conflict": train_df.loc[conflict, "attack_cat"].value_counts(),
}).assign(share=lambda t: (t["in_conflict"] / t["rows"]).round(3)).sort_values("share", ascending=False)

,rows,in_conflict,share
attack_cat,,,
DoS,12264,9746,0.795
Backdoor,1746,1340,0.767
Analysis,2000,1441,0.720
Exploits,33393,13299,0.398
Reconnaissance,10491,1889,0.180
Fuzzers,18184,2078,0.114
Worms,130,13,0.100
Generic,40000,319,0.008
Normal,56000,396,0.007


Rows per class after each policy (+ deduplicate):

In [38]:
counts = {"dedup only": deduplicate(train_df)["attack_cat"].value_counts()}
for target in ["label", "attack_cat"]:
    for policy in ["drop", "majority"]:
        resolved = deduplicate(resolve_conflicts(train_df, target, policy))
        counts[f"{target} / {policy}"] = resolved["attack_cat"].value_counts()

table = pd.DataFrame(counts)
table.loc["total"] = table.sum()
table

,dedup only,label / drop,label / majority,attack_cat / drop,attack_cat / majority
attack_cat,,,,,
Analysis,1594,1594,1594,393,393
Backdoor,1535,1535,1535,403,403
DoS,3806,3806,3806,2477,2691
Exploits,19844,19844,19844,18417,19227
Fuzzers,16150,15922,16004,14602,14684
Generic,4181,4181,4181,3904,3904
Normal,51890,51661,51709,51661,51709
Reconnaissance,7522,7522,7522,6209,6213
Shellcode,1091,1090,1090,1085,1086


Decision: `resolve_conflicts(..., policy="majority")` on the task's own target (`label` for binary, `attack_cat` for multi-class), then `deduplicate`. Test set is not touched.

On `attack_cat` most of Analysis, Backdoor and DoS go away with either policy, their flows mostly share feature vectors with other classes.

### Errors

In [39]:
try:
    load_unsw("validation")
except ValueError as e:
    print(e)

split must be one of ['train', 'test'], got 'validation'


In [40]:
try:
    load_unsw("train", data_dir="C:/nu_exista")
except FileNotFoundError as e:
    print(e)

C:\nu_exista\UNSW_NB15_training-set.csv not found. Download UNSW_NB15_training-set.csv from https://research.unsw.edu.au/projects/unsw-nb15-dataset and place it in C:\nu_exista.
